# Sequence Parallelism

在 Sequence Parallelism（序列并行，SP）场景下，通信与计算的重叠是提升大模型训练效率的关键技术。其核心思想是：在执行模型计算的同时，异步启动并执行必要的集合通信操作，从而隐藏通信延迟，避免 GPU 空闲等待。
## 一、All-Gather：聚合序列维度切片时，与 Attention 的 Q/K/V 投影计算重叠
- 背景：在 SP 中，输入序列被按长度切分到多个 GPU 上。每个 GPU 只持有部分序列的激活值。但在计算注意力机制时，需要完整的序列信息来生成 Query、Key、Value 向量。
- 通信操作：All-Gather —— 将所有 GPU 上的序列分片收集并拼接成完整序列。
- 计算操作：Attention 层中的线性投影（如 W_q, W_k, W_v 矩阵乘法）。
- 重叠策略：在 GPU 开始计算当前分片的 Q/K/V 投影之前或同时，启动 All-Gather 通信。由于 All-Gather 是异步的，它可以在后台进行数据传输，而 GPU 的计算单元可以同时处理本地分片的投影计算。当通信完成时，完整序列已就绪，可立即用于后续的注意力分数计算。
- 优势：避免了“先等所有数据传完再计算”的串行瓶颈，显著缩短了 Attention 层的总耗时。

## 二、Reduce-Scatter：分散序列梯度时，与反向传播的线性层计算重叠
- 背景：在反向传播阶段，每个 GPU 计算出自己负责部分的梯度后，需要将梯度汇总并平均分配到所有 GPU 上，以保证参数更新的正确性。
- 通信操作：Reduce-Scatter —— 对所有 GPU 的梯度进行归约（如求和或平均），然后将结果分散回各个 GPU。
- 计算操作：反向传播中的线性层梯度计算（如 dW = dY @ X^T）。
- 重叠策略：在完成当前层的梯度计算后，立即启动 Reduce-Scatter 通信。与此同时，GPU 可以开始计算下一层（或上一层，取决于反向传播顺序）的梯度。由于 Reduce-Scatter 的结果是后续参数更新所必需的，但其计算本身不依赖于当前层的梯度计算，因此可以安全地重叠。
- 优势：将原本串行的“计算梯度 -> 等待通信 -> 更新参数”流程，变为“计算梯度 + 启动通信 -> 计算下一层梯度 -> 等待通信完成 -> 更新参数”，有效利用了通信带宽。

## 三、技术实现基础

这种重叠依赖于现代深度学习框架（如 PyTorch）和硬件（如 NVIDIA GPU）提供的异步通信原语和 CUDA 流机制。通过将通信操作放入独立的 CUDA 流，并与计算流并发执行，可以实现真正的硬件级并行。




# 实现

在 Megatron-LM 中，Sequence Parallelism（SP）中的通信重叠机制主要实现在以下核心文件中：
- megatron/core/tensor_parallel/layers.py
  - 该文件定义了支持序列并行的线性层，如 ColumnParallelLinear 和 RowParallelLinear。在这些层的 forward 方法中，实现了 All-Gather 与 Q/K/V 投影计算的重叠逻辑。具体而言，在执行矩阵乘法前，代码会异步启动 All-Gather 操作，利用 CUDA 流的非阻塞特性，使通信与本地计算并行执行。
- megatron/core/tensor_parallel/mappings.py
  - 包含 gather_from_sequence_parallel_region 和 reduce_scatter_to_sequence_parallel_region 等函数。这些函数封装了 SP 所需的集合通信原语，并在内部处理了 Stream 切换与同步，确保通信操作不会阻塞主计算流。
- megatron/core/transformer/attention.py
  - 在注意力模块中，特别是使用 Ring Attention 时，该文件负责调度多轮次的局部计算与通信。它将序列切分后的 Q/K/V 块在 GPU 环上传递，并在每轮传递间隙插入计算任务，从而实现通信与计算的深度重叠。
- megatron/core/utils.py
  - 提供获取和管理 CUDA 流的工具函数，如 get_cuda_stream，用于创建或复用专用的通信流，是实现重叠调度的底层支撑。
  
这些文件共同协作，将 SP 场景下的 All-Gather 和 Reduce-Scatter 通信操作巧妙地嵌入到模型的前向与反向传播计算流程中，从而最大化 GPU 利用率，减少训练延迟。

